# Mapping social functioning profiles across childhood neurodevelopmental difficulties

**Self-organising map (SOM) and social profile classification**

Code accompanying: Trachtenberg, E., Mousley, A., Jelen, M., & Astle, D. *Mapping social functioning profiles across childhood neurodevelopmental difficulties: links with diagnosis, cognition, and everyday executive function.*

This notebook covers the SOM part of the pipeline (Figure 1):

1. Training the SOM on the four social dimension scores
2. Map quality (quantisation error and topographic error) and component planes (Figure 2A, Figure S2)
3. Simulated archetype profiles and one-vs-rest SVM classification of SOM neurons
4. Permutation-based island-size test to retain significant profile regions
5. Island centroids, categorical profile membership and continuous centroid-distance scores (Figure 2B)
6. Mean social dimension scores by profile

The network analysis used to derive the four social dimensions was run in R and is provided separately.

**Data availability.** No data are included in this repository; the code is shared to show the analysis. CALM data are available via the [CALM data portal](https://portal.camide.cam.ac.uk/overview/1158), subject to the cohort's data access procedures.

## 1. Setup

In [ ]:
import pickle
import platform
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib import colors
import seaborn as sns
from scipy.ndimage import label
from scipy.spatial import distance
from sklearn.svm import SVC
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
from minisom import MiniSom

warnings.filterwarnings("ignore", message=".*probability.*", category=FutureWarning)

In [ ]:
# ---- Paths ----
DATA_PATH = "socitems_mean_cluster_cleaned.csv"        # not included
PERMUTATION_PATH = "svm_island_permutation_results.pkl" # not included
OUTPUT_PATH = "participant_social_profiles.csv"         # not included

# ---- SOM parameters (Methods) ----
SEED = 42
GRID_SIZE = 10           # 10 x 10 rectangular grid
SIGMA = 2                # Gaussian neighbourhood
LEARNING_RATE = 0.5
N_ITERATIONS = 5000

# ---- Archetype simulation ----
N_SIMULATED = 500        # simulated observations per archetype
HIGH_PERCENTILE = 95     # focal dimension
BASE_PERCENTILE = 50     # remaining dimensions
COV_SCALE = 0.5          # shrinkage of the empirical covariance for sampling

# ---- Permutation island-size test ----
LOAD_SAVED_PERMUTATIONS = True  # True = reproduce published cut-offs; False = rerun
N_PERMUTATIONS = 1000
SIG_PERCENTILE = 99             # p < .01
CONNECTIVITY = np.ones((3, 3))  # 8-neighbour connectivity

In [ ]:
# Social dimensions (order of the input columns)
DIMENSIONS = ["Friendship difficulties", "Social withdrawal",
              "Peer victimisation", "Prosocial behaviour"]

RAW_COLUMNS = {"Cluster_1": "Friendship difficulties",
               "Cluster_2": "Social withdrawal",
               "Cluster_3": "Peer victimisation",
               "Cluster_4": "Prosocial behaviour"}

# Archetypes: key -> index of the focal dimension (None = null reference)
# Order matters for reproducibility of the simulated data
ARCHETYPES = {"prosocial": 3,
              "friendship": 0,
              "withdrawal": 1,
              "victimisation": 2,
              "null": None}

PROFILE_NAMES = {"prosocial": "Prosocial behaviour",
                 "friendship": "Friendship difficulties",
                 "withdrawal": "Social withdrawal",
                 "victimisation": "Peer victimisation",
                 "null": "Null (median)",
                 "unassigned": "Unassigned"}

PROFILE_COLOURS = {"prosocial": "#1f77b4",
                   "friendship": "#d62728",
                   "withdrawal": "#ffbf00",
                   "victimisation": "#8c564b",
                   "null": "#888888",
                   "unassigned": "#888888"}

# Keys used in the saved permutation file
LEGACY_KEYS = {"high_prosocial": "prosocial",
               "high_making_friends": "friendship",
               "high_social_difficulty": "withdrawal",
               "high_rejection_bullying": "victimisation",
               "high_null": "null"}

ARCHETYPE_KEYS = list(ARCHETYPES)                          # includes null
PROFILE_KEYS = [k for k in ARCHETYPE_KEYS if k != "null"]  # four retained profiles

plt.rcParams.update({"figure.dpi": 150, "font.size": 11,
                     "font.family": "sans-serif", "axes.titlesize": 13})

## 2. Load the social dimension scores

In [ ]:
social = pd.read_csv(DATA_PATH).rename(columns=RAW_COLUMNS)
data_for_som = social[DIMENSIONS].values

print(f"N = {len(social)}")
print(f"Score range: {data_for_som.min():.2f} to {data_for_som.max():.2f}")
social[DIMENSIONS].describe().round(2)

In [ ]:
plt.figure(figsize=(6, 5))
sns.heatmap(social[DIMENSIONS].corr(), annot=True, fmt=".2f",
            cmap="coolwarm", vmin=-1, vmax=1, square=True)
plt.title("Correlations between social dimensions")
plt.tight_layout()
plt.show()

## 3. Self-organising map

The SOM (Kohonen, 2001) was trained on a 10×10 rectangular grid with Euclidean activation distance in MiniSom, for 5,000 iterations with random initialisation, a Gaussian neighbourhood function (sigma = 2) and learning rate 0.5. Each child was mapped to their best matching unit (BMU), the node whose weight vector was closest to the child's input vector. Map quality was summarised using quantisation error (QE) and topographic error (TE). See Shalev et al. (2025) for a practical guide.

In [ ]:
np.random.seed(SEED)

som = MiniSom(GRID_SIZE, GRID_SIZE, len(DIMENSIONS),
              sigma=SIGMA,
              learning_rate=LEARNING_RATE,
              topology="rectangular",
              neighborhood_function="gaussian",
              activation_distance="euclidean",
              random_seed=SEED)

som.random_weights_init(data_for_som)
som.train_random(data_for_som, N_ITERATIONS, verbose=False)

weights = som.get_weights()                                   # (10, 10, 4)
bmu_coords = np.array([som.winner(x) for x in data_for_som])  # (row, col) per child

qe = som.quantization_error(data_for_som)
te = som.topographic_error(data_for_som)
print(f"Quantisation error (QE): {qe:.3f}")
print(f"Topographic error (TE):  {te:.3f}")

### Distance map (U-matrix) and activation map (Figure S2)

In [ ]:
u_matrix = som.distance_map()
activations = som.activation_response(data_for_som)
ticks = np.arange(GRID_SIZE)

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
panels = [(u_matrix, "Distance map (U-matrix)", None),
          (activations, "Activation map (BMU frequencies)", np.percentile(activations, 99))]

for ax, (values, title, vmax) in zip(axes, panels):
    im = ax.imshow(values, cmap="viridis", vmin=0 if vmax else None, vmax=vmax)
    ax.set_title(title)
    ax.set_xticks(ticks, ticks + 1)
    ax.set_yticks(ticks, ticks + 1)
    ax.set_xlabel("SOM x-coordinate")
    ax.set_ylabel("SOM y-coordinate")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

fig.tight_layout()
plt.show()

print(f"Nodes with no BMU hits: {int(np.sum(activations == 0))}")

### Component planes (Figure 2A)

Each SOM node stores a four-value weight vector, one value per social dimension. Component planes plot these values one dimension at a time.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4), constrained_layout=True)
vmin, vmax = weights.min(), weights.max()   # shared scale

for i, ax in enumerate(axes):
    im = ax.imshow(weights[:, :, i], cmap="viridis", vmin=vmin, vmax=vmax)
    ax.set_title(DIMENSIONS[i])
    ax.set_xticks([])
    ax.set_yticks([])
    fig.colorbar(im, ax=ax, fraction=0.05, pad=0.025)

plt.show()

## 4. Simulated archetype profiles

To identify interpretable regions within the graded map, we used a supervised classification approach anchored by simulated archetype profiles rather than standard clustering. Four archetypes were generated to reflect prosocial behaviour, friendship difficulties, social withdrawal and peer victimisation. For each archetype, the corresponding dimension was set to the 95th percentile and the remaining dimensions to the 50th percentile. A null archetype reflecting median values across all dimensions was included as a reference. 500 observations were simulated for each archetype from a multivariate normal distribution using the empirical covariance (scaled by 0.5) and clipped to the 0-1 range.

In [ ]:
rng = np.random.default_rng(seed=SEED)

p_high = np.percentile(data_for_som, HIGH_PERCENTILE, axis=0)
p_base = np.percentile(data_for_som, BASE_PERCENTILE, axis=0)
cov_sim = COV_SCALE * np.cov(data_for_som, rowvar=False)

archetype_means = {}
for key, focal in ARCHETYPES.items():
    mean_vec = p_base.copy()
    if focal is not None:
        mean_vec[focal] = p_high[focal]
    archetype_means[key] = mean_vec

archetype_data = {key: np.clip(rng.multivariate_normal(mean_vec, cov_sim, size=N_SIMULATED), 0, 1)
                  for key, mean_vec in archetype_means.items()}

(pd.DataFrame(archetype_means, index=DIMENSIONS).T
   .rename(index=PROFILE_NAMES).round(2))

### Where the simulated archetypes land on the map

In [ ]:
archetype_bmus = {key: np.array([som.winner(x) for x in sims])
                  for key, sims in archetype_data.items()}

fig, axes = plt.subplots(1, len(ARCHETYPE_KEYS), figsize=(4 * len(ARCHETYPE_KEYS), 4))
jitter = 0.25

for ax, key in zip(axes, ARCHETYPE_KEYS):
    bmus = archetype_bmus[key]
    ax.imshow(u_matrix, cmap="viridis")
    ax.scatter(bmus[:, 1] + (rng.random(len(bmus)) - 0.5) * jitter,
               bmus[:, 0] + (rng.random(len(bmus)) - 0.5) * jitter,
               c="red", s=20, marker="x", alpha=0.6, linewidths=0.8)
    n_nodes = np.unique(bmus, axis=0).shape[0]
    ax.set_title(f"{PROFILE_NAMES[key]}\n({n_nodes} nodes hit)")
    ax.axis("off")

plt.tight_layout()
plt.show()

## 5. One-vs-rest SVM classification of SOM neurons

One-vs-rest support vector machine (SVM) classifiers with a radial basis function kernel (Cortes & Vapnik, 1995) were trained on the simulated archetype observations and applied to the SOM neuron weight vectors. Each neuron was assigned to the archetype with the highest decision score.

In [ ]:
X_train = np.vstack([archetype_data[k] for k in ARCHETYPE_KEYS])
y_train = np.concatenate([[k] * N_SIMULATED for k in ARCHETYPE_KEYS])

svms = {}
for key in ARCHETYPE_KEYS:
    clf = SVC(kernel="rbf", probability=True, class_weight="balanced", random_state=SEED)
    clf.fit(X_train, (y_train == key).astype(int))
    svms[key] = clf

def classify_neurons(neuron_weights):
    # Returns the index (in ARCHETYPE_KEYS) of the highest-scoring SVM for each neuron
    scores = np.column_stack([svms[k].decision_function(neuron_weights) for k in ARCHETYPE_KEYS])
    return np.argmax(scores, axis=1)

neuron_weights = weights.reshape(-1, weights.shape[2])     # (100, 4)
svm_map = classify_neurons(neuron_weights).reshape(GRID_SIZE, GRID_SIZE)

In [ ]:
archetype_cmap = colors.ListedColormap([PROFILE_COLOURS[k] for k in ARCHETYPE_KEYS])

fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(svm_map, cmap=archetype_cmap, vmin=0, vmax=len(ARCHETYPE_KEYS) - 1)
patches = [mpatches.Patch(color=PROFILE_COLOURS[k], label=PROFILE_NAMES[k]) for k in ARCHETYPE_KEYS]
ax.legend(handles=patches, bbox_to_anchor=(1.02, 1), loc="upper left", title="SVM winning profile")
ax.set_title("SOM neurons classified by one-vs-rest SVM")
ax.axis("off")
plt.tight_layout()
plt.show()

## 6. Permutation-based island-size test

We tested whether same-label regions formed spatially contiguous islands larger than expected by chance. Islands were defined using 8-neighbour connectivity. Null island-size distributions were generated from 1,000 permutations in which SOM neuron weight vectors were randomly reassigned to grid locations and reclassified. For each archetype, the minimum island size required for significance was the 99th percentile of its null distribution (p < .01).

The SVMs are trained once on the simulated data, so reclassifying the shuffled weights with the same classifiers is equivalent to retraining them in every permutation.

Set `LOAD_SAVED_PERMUTATIONS = True` to load the null distributions used in the paper. Rerunning the permutations gives very similar, but not necessarily identical, cut-offs.

In [ ]:
def island_sizes(class_map, idx):
    labelled, n = label(class_map == idx, structure=CONNECTIVITY)
    return np.bincount(labelled.ravel())[1:] if n > 0 else np.array([], dtype=int)

observed_sizes = {k: island_sizes(svm_map, i) for i, k in enumerate(ARCHETYPE_KEYS)}

if LOAD_SAVED_PERMUTATIONS:
    with open(PERMUTATION_PATH, "rb") as f:
        saved = pickle.load(f)
    null_sizes = {LEGACY_KEYS[k]: np.asarray(v) for k, v in saved["null_dists_by_profile_svm"].items()}

    # Check the saved map matches the map produced above
    saved_map = np.array(saved["numeric_map_svm"])
    legacy_order = [LEGACY_KEYS[k] for k in saved["all_profile_names"]]
    remap = np.array([ARCHETYPE_KEYS.index(k) for k in legacy_order])
    print("SVM map identical to saved map:", np.array_equal(remap[saved_map], svm_map))
else:
    perm_rng = np.random.default_rng(SEED)
    null_sizes = {k: [] for k in ARCHETYPE_KEYS}
    for p in range(N_PERMUTATIONS):
        shuffled = neuron_weights[perm_rng.permutation(len(neuron_weights))]
        perm_map = classify_neurons(shuffled).reshape(GRID_SIZE, GRID_SIZE)
        for i, k in enumerate(ARCHETYPE_KEYS):
            null_sizes[k].extend(island_sizes(perm_map, i))
    null_sizes = {k: np.asarray(v) for k, v in null_sizes.items()}

cutoffs = {k: int(np.ceil(np.percentile(v, SIG_PERCENTILE))) if len(v) else np.inf
           for k, v in null_sizes.items()}

pd.DataFrame({"Observed island sizes": [sorted(map(int, observed_sizes[k]), reverse=True) for k in ARCHETYPE_KEYS],
              f"Cut-off ({SIG_PERCENTILE}th percentile)": [cutoffs[k] for k in ARCHETYPE_KEYS]},
             index=[PROFILE_NAMES[k] for k in ARCHETYPE_KEYS])

In [ ]:
fig, axes = plt.subplots(1, len(ARCHETYPE_KEYS), figsize=(4 * len(ARCHETYPE_KEYS), 3.5), sharey=True)
bins = np.arange(1, 22) - 0.5

for ax, k in zip(axes, ARCHETYPE_KEYS):
    ax.hist(null_sizes[k], bins=bins, color="#888888", edgecolor="black")
    ax.axvline(cutoffs[k], color="red", linestyle="--", label="Cut-off")
    ax.set_yscale("log")
    ax.set_title(PROFILE_NAMES[k])
    ax.set_xlabel("Island size (neurons)")
axes[0].set_ylabel("Count")
axes[0].legend()
plt.suptitle("Null distributions of island sizes")
plt.tight_layout()
plt.show()

## 7. Significant profile regions and centroids (Figure 2B)

Significant islands were retained for the four archetypal profiles; the null archetype served as a reference only. Island centroids were computed as the mean row and column coordinates of neurons within each island.

In [ ]:
significant_map = -np.ones((GRID_SIZE, GRID_SIZE), dtype=int)   # -1 = outside all retained regions
centroids = {k: [] for k in PROFILE_KEYS}

for k in PROFILE_KEYS:
    idx = ARCHETYPE_KEYS.index(k)
    labelled, n = label(svm_map == idx, structure=CONNECTIVITY)
    for island in range(1, n + 1):
        cells = np.argwhere(labelled == island)
        if len(cells) >= cutoffs[k]:
            significant_map[labelled == island] = idx
            centroids[k].append(cells.mean(axis=0))   # (row, col)

for k in PROFILE_KEYS:
    cents = ", ".join(f"({r:.2f}, {c:.2f})" for r, c in centroids[k])
    print(f"{PROFILE_NAMES[k]}: {len(centroids[k])} island(s), centroid (row, col) = {cents}")

In [ ]:
background = "#f0f0f0"
region_cmap = colors.ListedColormap([background] + [PROFILE_COLOURS[k] for k in ARCHETYPE_KEYS])

fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(significant_map + 1, cmap=region_cmap, vmin=0, vmax=len(ARCHETYPE_KEYS))

for k in PROFILE_KEYS:
    for r, c in centroids[k]:
        ax.scatter(c, r, s=400, marker="*", color=PROFILE_COLOURS[k],
                   edgecolor="k", linewidth=2, zorder=10)
        ax.text(c + 0.3, r + 0.3, PROFILE_NAMES[k], fontsize=13,
                fontweight="bold", va="bottom", ha="left", zorder=11)

ticks = np.arange(GRID_SIZE)
ax.set_xticks(ticks, ticks + 1)
ax.set_yticks(ticks, ticks + 1)
ax.set_xlabel("SOM x-coordinate")
ax.set_ylabel("SOM y-coordinate")
ax.set_title("Significant profile regions and centroids")
plt.tight_layout()
plt.show()

## 8. Profile outputs

**Categorical profile membership.** Children were assigned to the profile island containing their BMU. Children whose BMU fell outside all significant islands were classified as unassigned.

**Continuous profile expression.** Euclidean distance was computed between each child's BMU coordinate and the centroid of each significant island (minimum distance if a profile had more than one island). Lower values indicate greater proximity to, and stronger expression of, the corresponding profile.

In [ ]:
region_at_bmu = significant_map[bmu_coords[:, 0], bmu_coords[:, 1]]

profiles = pd.DataFrame({"id": social["id"].values,
                         "bmu_row": bmu_coords[:, 0],
                         "bmu_col": bmu_coords[:, 1],
                         "profile": [ARCHETYPE_KEYS[i] if i >= 0 else "unassigned"
                                     for i in region_at_bmu]})

for k in PROFILE_KEYS:
    profiles[f"dist_{k}"] = [round(min(distance.euclidean(bmu, c) for c in centroids[k]), 1)
                             if centroids[k] else np.nan
                             for bmu in bmu_coords]

# Profile sizes (Table 1)
(profiles["profile"].map(PROFILE_NAMES).value_counts()
   .rename("n").to_frame())

In [ ]:
# Participant-level output for the downstream analyses (keep local, do not upload)
profiles.to_csv(OUTPUT_PATH, index=False)
print(f"Saved {OUTPUT_PATH}: {profiles.shape[0]} rows, columns = {list(profiles.columns)}")

## 9. Social dimension scores by profile

Mean scores across the four social dimensions were compared across BMU-assigned profile groups using one-way ANOVA, with Benjamini-Hochberg FDR correction across the four tests.

In [ ]:
merged = profiles[["id", "profile"]].merge(social[["id"] + DIMENSIONS], on="id")
merged["Profile"] = merged["profile"].map(PROFILE_NAMES)

(merged.groupby("Profile")[DIMENSIONS]
   .agg(["mean", "sem"]).round(2))

In [ ]:
anova_rows = []
for dim in DIMENSIONS:
    model = smf.ols(f"Q('{dim}') ~ C(Profile)", data=merged).fit()
    aov = sm.stats.anova_lm(model, typ=2)
    anova_rows.append({"Dimension": dim,
                       "df": f"{int(aov.loc['C(Profile)', 'df'])}, {int(aov.loc['Residual', 'df'])}",
                       "F": aov.loc["C(Profile)", "F"],
                       "p": aov.loc["C(Profile)", "PR(>F)"],
                       "eta_sq": aov.loc["C(Profile)", "sum_sq"] / aov["sum_sq"].sum()})

anova = pd.DataFrame(anova_rows)
anova["p_FDR"] = multipletests(anova["p"], method="fdr_bh")[1]
anova.round({"F": 2, "eta_sq": 2})

In [ ]:
order = [PROFILE_NAMES[k] for k in PROFILE_KEYS + ["unassigned"]
         if PROFILE_NAMES[k] in set(merged["Profile"])]
plot_colours = {PROFILE_NAMES[k]: PROFILE_COLOURS[k] for k in PROFILE_KEYS + ["unassigned"]}
means = merged.groupby("Profile")[DIMENSIONS].mean().loc[order]

angles = np.linspace(0, 2 * np.pi, len(DIMENSIONS), endpoint=False)
angles = np.concatenate([angles, angles[:1]])

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
for prof in order:
    values = np.concatenate([means.loc[prof].values, means.loc[prof].values[:1]])
    ax.plot(angles, values, color=plot_colours[prof], linewidth=2, label=prof)
    ax.fill(angles, values, color=plot_colours[prof], alpha=0.07)

ax.set_thetagrids(np.degrees(angles[:-1]), labels=DIMENSIONS)
ax.tick_params(axis="x", pad=15)
ax.set_ylim(0, 1)
ax.set_title("Mean social dimension scores by profile", pad=25)
ax.legend(loc="center left", bbox_to_anchor=(1.15, 0.5), frameon=False, title="Profile")
plt.tight_layout()
plt.show()

## Software versions

In [ ]:
from importlib.metadata import version
print("Python", platform.python_version())
for pkg in ["numpy", "pandas", "MiniSom", "scikit-learn", "scipy", "statsmodels", "matplotlib", "seaborn"]:
    print(f"{pkg}: {version(pkg)}")

## References

Cortes, C., & Vapnik, V. (1995). Support-vector networks. *Machine Learning, 20*, 273-297. https://doi.org/10.1007/BF00994018

Kohonen, T. (2001). *Self-organizing maps* (3rd ed.). Springer. https://doi.org/10.1007/978-3-642-56927-2

Shalev, I., Trachtenberg, E., Jelen, M., Mills, W., Bennett, M., Lawson, R., Dalgleish, T., & Astle, D. (2025). A practical guide to self-organising maps in psychological research. *PsyArXiv*. https://osf.io/preprints/psyarxiv/dy6z2